In [31]:
from interface import *
from visualize import *

import random
import stim
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=np.inf) # type: ignore

import plotly.io as pio
pio.renderers.default = 'notebook_connected'


%load_ext autoreload
%aimport interface.chip
%aimport interface.models
%aimport visualize.visualize
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
list(pio.renderers)

['plotly_mimetype',
 'jupyterlab',
 'nteract',
 'vscode',
 'notebook',
 'notebook_connected',
 'kaggle',
 'azure',
 'colab',
 'cocalc',
 'databricks',
 'json',
 'png',
 'jpeg',
 'jpg',
 'svg',
 'pdf',
 'browser',
 'firefox',
 'chrome',
 'chromium',
 'iframe',
 'iframe_connected',
 'sphinx_gallery',
 'sphinx_gallery_png']

## Basic Chip Demo

Creating a basic l=5 x h=5 chip results in an underlying grid structure ranging from (0,0) to (2l, 2h). The chip is in a checkerboard style, where valid (x, y) coords have the property of x%2 == y%2. The chip can be visualized by using `chip.show()`

In [33]:
chip = Chip(5,5)
chip.show()

In [34]:
chip.loc((0,0)).status = Status.LOGICAL
chip.loc((5,5)).status = Status.ANCILLA
print(chip.loc((0,0)))
print(chip.loc((5,5)))

Qubit(loc=(0, 0), status=Status.LOGICAL, noise=NoiseProfile(p=0.0))
Qubit(loc=(5, 5), status=Status.ANCILLA, noise=NoiseProfile(p=0.0))


In [35]:
chip.show(style=default_style)

In [36]:
chip.generate_random_noise()
chip.loc((2, 2)).noise.p = .06
print(chip.loc((2, 2)))
print(chip.noise_map)

Qubit(loc=(2, 2), status=Status.INACTIVE, noise=NoiseProfile(p=0.06))
{(0, 0): NoiseProfile(p=0.02946), (0, 2): NoiseProfile(p=0.02083), (0, 4): NoiseProfile(p=0.0404), (0, 6): NoiseProfile(p=0.04239), (0, 8): NoiseProfile(p=0.02175), (1, 1): NoiseProfile(p=0.02994), (1, 3): NoiseProfile(p=0.03715), (1, 5): NoiseProfile(p=0.02203), (1, 7): NoiseProfile(p=0.04089), (1, 9): NoiseProfile(p=0.03147), (2, 0): NoiseProfile(p=0.04533), (2, 2): NoiseProfile(p=0.06), (2, 4): NoiseProfile(p=0.02431), (2, 6): NoiseProfile(p=0.03454), (2, 8): NoiseProfile(p=0.01264), (3, 1): NoiseProfile(p=0.01742), (3, 3): NoiseProfile(p=0.04307), (3, 5): NoiseProfile(p=0.01399), (3, 7): NoiseProfile(p=0.03444), (3, 9): NoiseProfile(p=0.04738), (4, 0): NoiseProfile(p=0.04145), (4, 2): NoiseProfile(p=0.02208), (4, 4): NoiseProfile(p=0.04732), (4, 6): NoiseProfile(p=0.0248), (4, 8): NoiseProfile(p=0.02871), (5, 1): NoiseProfile(p=0.01996), (5, 3): NoiseProfile(p=0.02005), (5, 5): NoiseProfile(p=0.01266), (5, 7): No

In [37]:
chip.show(style=noise_heatmap_style(chip, limits =(0.0, 0.06)))

## Logical Tile Placement and Shifts

Chips are a workspace for tiles, that represent logical qubits created from STIM circuits. To create a tile, provide a STIM circuit. The size and qubit coordinates are automatically pulled from the file. Once a chip has been instantiated, you can add tiles for a given coordinate, which represents the upper leftmost qubit of the tile.

In [38]:
chip = Chip(7,7)
tile = LogicalTile(circuit=stim.Circuit.generated(code_task='surface_code:rotated_memory_z', distance=5, rounds=3), tag=TileTag(distance=5))
chip.add_tile(tile, loc=(0,0))
chip.generate_random_noise()
chip.show()

In [39]:
chip.show(style=noise_heatmap_style(chip, limits=(0.0,0.05)))

In [40]:
chip.tiles[0].qubits

[Qubit(loc=(2, 0), status=Status.LOGICAL, noise=NoiseProfile(p=0.03957)),
 Qubit(loc=(0, 0), status=Status.ANCILLA, noise=NoiseProfile(p=0.02437)),
 Qubit(loc=(3, 1), status=Status.LOGICAL, noise=NoiseProfile(p=0.02219)),
 Qubit(loc=(1, 1), status=Status.LOGICAL, noise=NoiseProfile(p=0.02309)),
 Qubit(loc=(0, 2), status=Status.ANCILLA, noise=NoiseProfile(p=0.02823)),
 Qubit(loc=(2, 2), status=Status.LOGICAL, noise=NoiseProfile(p=0.02508)),
 Qubit(loc=(3, 3), status=Status.LOGICAL, noise=NoiseProfile(p=0.01163)),
 Qubit(loc=(1, 3), status=Status.LOGICAL, noise=NoiseProfile(p=0.02094)),
 Qubit(loc=(0, 4), status=Status.LOGICAL, noise=NoiseProfile(p=0.02851)),
 Qubit(loc=(2, 4), status=Status.LOGICAL, noise=NoiseProfile(p=0.02455)),
 Qubit(loc=(4, 0), status=Status.ANCILLA, noise=NoiseProfile(p=0.03081)),
 Qubit(loc=(6, 0), status=Status.LOGICAL, noise=NoiseProfile(p=0.04664)),
 Qubit(loc=(5, 1), status=Status.LOGICAL, noise=NoiseProfile(p=0.01951)),
 Qubit(loc=(7, 1), status=Status.LOGIC

In [41]:
chip.tiles[0].shift_by(2,0)
chip.show()
print(chip.tiles[0].origin)

(2.0, 0.0)


In [42]:
chip.tiles[0].shift_by(0, 2)
chip.show()
print(chip.tiles[0].origin)

(2.0, 2.0)


In [43]:
chip.tiles[0].shift_by(0, -2)
chip.show()
print(chip.tiles[0].origin)

(2.0, 0.0)


In [44]:
chip.tiles[0].shift_by(-2, 0)
chip.show()
print(chip.tiles[0].origin)

(0.0, 0.0)


In [45]:
chip.tiles[0].shift_to((2,2))
chip.show()

In [46]:
chip.tiles[0].shift_to((0,0))
chip.show()

## Multi-Tile Demo

In [47]:
tile3 = LogicalTile(circuit=stim.Circuit.generated(code_task='surface_code:rotated_memory_z', distance=5, rounds=3), tag=TileTag(distance=5))
chip3 = Chip(12, 12)
chip3.generate_random_noise((0.0,0.05))
chip3.add_tile(tile3,loc=(0,0))
chip3.show()

In [48]:
chip3.add_tile(tile3.copy(),loc=(12,0))
chip3.add_tile(tile3.copy(),loc=(0,12))
chip3.add_tile(tile3.copy(),loc=(12,12))

True

In [49]:
chip3.show()

In [50]:
chip3.show(style=noise_heatmap_style(chip3, limits=(0.0, 0.05)))

## Multi Tile With Various Distances

In [ ]:
d3Tile = LogicalTile(circuit=stim.Circuit.generated(code_task='surface_code:rotated_memory_z', distance=3, rounds=3), tag=TileTag(distance=3))
d5Tile = LogicalTile(circuit=stim.Circuit.generated(code_task='surface_code:rotated_memory_z', distance=5, rounds=3), tag=TileTag(distance=5))
d7Tile = LogicalTile(circuit=stim.Circuit.generated(code_task='surface_code:rotated_memory_z', distance=7, rounds=3), tag=TileTag(distance=7))

chip4 = Chip(14, 14)
chip4.generate_random_noise((0.0,0.05))
chip4.add_tile(d3Tile.copy(),loc=(0,0))

chip4.add_tile(d3Tile.copy(),loc=(8,0))
chip4.add_tile(d3Tile.copy(),loc=(16,0))

chip4.add_tile(d5Tile.copy(),loc=(0,8))

chip4.add_tile(d7Tile.copy(),loc=(12,8))
chip4.show()

In [52]:
chip4.show(noise_heatmap_style(chip4, limits=(0.0, 0.05)))